# Bloque IV — Clustering, silhouette y PCA

**Duración estimada:** 3 horas  
**Dataset:** `../data/segmentacion_clientes_mayo_2026.csv`

## Objetivo de aprendizaje

El alumnado aprenderá a aplicar clustering para segmentación, seleccionar un número razonable de grupos, evaluar la calidad mediante silhouette y visualizar los resultados con PCA.

## Agenda de 3 horas

| Tiempo | Actividad |
|---:|---|
| 0:00–0:25 | Aprendizaje no supervisado |
| 0:25–0:55 | Escalado y preparación |
| 0:55–1:25 | K-Means y método del codo |
| 1:25–1:35 | Pausa |
| 1:35–2:05 | Silhouette score |
| 2:05–2:35 | PCA para visualización |
| 2:35–3:00 | Perfilado de clusters |

In [ ]:
# Configuración común
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## 1. Carga del dataset

En clustering no tenemos variable objetivo. Queremos descubrir grupos de clientes con comportamientos similares.

In [ ]:
df = pd.read_csv("../data/segmentacion_clientes_mayo_2026.csv")
df.head()

## 2. Selección de variables

Elegimos variables numéricas que describen comportamiento, valor y relación comercial.

In [ ]:
features = [
    "ingresos",
    "compras_12m",
    "ticket_medio",
    "visitas_web",
    "dias_desde_ultima_compra",
    "reclamaciones"
]

X = df[features]
X.describe()

## 3. Escalado

K-Means se basa en distancias, por lo que las variables deben estar en escalas comparables.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=features).describe()

## 4. K-Means inicial

Probamos inicialmente con 3 clusters para entender el flujo completo.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df["cluster"] = clusters
df["cluster"].value_counts()

## 5. Método del codo

La inercia mide la compactación interna de los clusters. Al aumentar k, la inercia baja, pero buscamos un punto donde la mejora marginal se reduzca.

In [ ]:
inertias = []

k_values = range(2, 11)

for k in k_values:
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    modelo.fit(X_scaled)
    inertias.append(modelo.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Número de clusters")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.show()

## 6. Silhouette score

Silhouette combina cohesión y separación. Valores cercanos a 1 indican clusters más claros.

In [ ]:
silhouettes = []

for k in k_values:
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = modelo.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouettes.append(score)

sil_df = pd.DataFrame({
    "k": list(k_values),
    "silhouette": silhouettes
})

sil_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(sil_df["k"], sil_df["silhouette"], marker="o")
plt.xlabel("Número de clusters")
plt.ylabel("Silhouette score")
plt.title("Evaluación de clusters por silhouette")
plt.show()

## 7. Modelo final

Seleccionamos el número de clusters. En un proyecto real, la decisión combina métrica, interpretabilidad y utilidad de negocio.

In [ ]:
k_final = int(sil_df.sort_values("silhouette", ascending=False).iloc[0]["k"])
print("k seleccionado por silhouette:", k_final)

modelo_final = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df["cluster"] = modelo_final.fit_predict(X_scaled)

df["cluster"].value_counts()

## 8. PCA para visualización

PCA reduce la dimensionalidad a dos componentes para representar los grupos en un plano.

In [ ]:
pca = PCA(n_components=2)
componentes = pca.fit_transform(X_scaled)

df["PC1"] = componentes[:, 0]
df["PC2"] = componentes[:, 1]

print("Varianza explicada:", pca.explained_variance_ratio_)
print("Varianza explicada acumulada:", pca.explained_variance_ratio_.sum())

In [ ]:
plt.figure(figsize=(7, 5))
for c in sorted(df["cluster"].unique()):
    subset = df[df["cluster"] == c]
    plt.scatter(subset["PC1"], subset["PC2"], label=f"Cluster {c}", alpha=0.7)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Clusters visualizados con PCA")
plt.legend()
plt.show()

## 9. Perfilado de clusters

El verdadero valor del clustering está en interpretar los grupos y traducirlos en acciones.

In [ ]:
perfil_clusters = (
    df
    .groupby("cluster")[features]
    .mean()
    .round(2)
)

perfil_clusters

In [ ]:
for cluster_id, row in perfil_clusters.iterrows():
    print(f"Cluster {cluster_id}")
    print(row.sort_values(ascending=False))
    print("-" * 50)

## 10. Propuesta de acciones

Ejemplo de lectura:

- Clientes con alto ticket y alta frecuencia: fidelización premium.
- Clientes con muchas visitas y pocas compras: campañas de conversión.
- Clientes con muchos días desde última compra: reactivación.
- Clientes con reclamaciones elevadas: mejora de experiencia.

## 11. Ejercicio integrador

1. Prueba K-Means con `k=2`, `k=3`, `k=4` y `k=5`.
2. Compara silhouette.
3. Visualiza cada solución con PCA.
4. Perfila los clusters de la mejor solución.
5. Asigna un nombre de negocio a cada cluster.
6. Propón una acción comercial para cada segmento.

### Entregable

Notebook con tabla de perfiles y conclusión por cluster.

In [ ]:
# Espacio para el ejercicio del alumnado